# Proyecto Final - Herramienta de Visualización de Datos de COVID-19
## Steven guerra
## Alberto bellera
## Juan cortes
## Neel patel
## Samuel castro
## Yeshua Zapata

In [1]:
import pandas as pd
import numpy as np
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px

In [2]:
# Importamos el dataset

df = pd.read_csv("./owid-covid-data.csv")

df['total_cases'] = df.groupby('location')['total_cases'].transform('ffill')
df['total_deaths'] = df.groupby('location')['total_deaths'].transform('ffill')
df['people_vaccinated'] = df.groupby('location')['people_vaccinated'].transform('ffill')
df['people_vaccinated_per_hundred'] = df.groupby('location')['people_vaccinated_per_hundred'].transform('ffill')

df['total_cases_cumsum'] = df.groupby('location')['total_cases'].cummax()
df['total_deaths_cumsum'] = df.groupby('location')['total_deaths'].cummax()
df['people_vaccinated_cumsum'] = df.groupby('location')['people_vaccinated'].cummax()

In [3]:
df.head(1000)

,iso_code,continent,location,date,total_cases,new_cases,new_cases_smoothed,total_deaths,new_deaths,new_deaths_smoothed,...,life_expectancy,human_development_index,population,excess_mortality_cumulative_absolute,excess_mortality_cumulative,excess_mortality,excess_mortality_cumulative_per_million,total_cases_cumsum,total_deaths_cumsum,people_vaccinated_cumsum
0,AFG,Asia,Afghanistan,2020-01-05,NaN,0.0,NaN,NaN,0.0,NaN,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,AFG,Asia,Afghanistan,2020-01-06,NaN,0.0,NaN,NaN,0.0,NaN,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,AFG,Asia,Afghanistan,2020-01-07,NaN,0.0,NaN,NaN,0.0,NaN,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,AFG,Asia,Afghanistan,2020-01-08,NaN,0.0,NaN,NaN,0.0,NaN,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AFG,Asia,Afghanistan,2020-01-09,NaN,0.0,NaN,NaN,0.0,NaN,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,AFG,Asia,Afghanistan,2022-09-26,198283.0,0.0,171.571,7796.0,0.0,0.286,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,198283.0,7796.0,10917914.0
996,AFG,Asia,Afghanistan,2022-09-27,198283.0,0.0,171.571,7796.0,0.0,0.286,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,198283.0,7796.0,10917914.0
997,AFG,Asia,Afghanistan,2022-09-28,198283.0,0.0,171.571,7796.0,0.0,0.286,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,198283.0,7796.0,10929549.0
998,AFG,Asia,Afghanistan,2022-09-29,198283.0,0.0,171.571,7796.0,0.0,0.286,...,64.83,0.511,41128772.0,NaN,NaN,NaN,NaN,198283.0,7796.0,10929549.0


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404525 entries, 0 to 404524
Data columns (total 70 columns):
 #   Column                                      Non-Null Count   Dtype  
---  ------                                      --------------   -----  
 0   iso_code                                    404525 non-null  object 
 1   continent                                   385141 non-null  object 
 2   location                                    404525 non-null  object 
 3   date                                        404525 non-null  object 
 4   total_cases                                 365721 non-null  float64
 5   new_cases                                   393220 non-null  float64
 6   new_cases_smoothed                          391990 non-null  float64
 7   total_deaths                                343091 non-null  float64
 8   new_deaths                                  393598 non-null  float64
 9   new_deaths_smoothed                         392368 non-null  float64
 

In [5]:
df.isnull().sum()

iso_code                                        0
continent                                   19384
location                                        0
date                                            0
total_cases                                 38804
                                            ...  
excess_mortality                           391247
excess_mortality_cumulative_per_million    391247
total_cases_cumsum                          38804
total_deaths_cumsum                         61434
people_vaccinated_cumsum                   126701
Length: 70, dtype: int64

In [6]:
# Seleccionar las columnas relevantes
columns_to_keep = [
    'iso_code', 'continent', 'location', 'date', 'total_cases', 'new_cases_smoothed', 'total_deaths', 'new_deaths_smoothed', 'people_vaccinated',
    'new_people_vaccinated_smoothed',
    'total_cases_per_million',
    'new_cases_per_million', 'reproduction_rate', 'total_deaths_per_million',
    'new_deaths_per_million', 'icu_patients_per_million', 'hosp_patients_per_million',
    'total_tests_per_thousand', 'new_tests_per_thousand', 'positive_rate',
    'tests_per_case', 'tests_units', 'total_vaccinations_per_hundred',
    'people_vaccinated_per_hundred', 'people_fully_vaccinated_per_hundred',
    'total_boosters_per_hundred', 'population', 'population_density',
    'median_age', 'gdp_per_capita', 'life_expectancy', 'human_development_index',
    'stringency_index', 'aged_65_older', 'aged_70_older', 'cardiovasc_death_rate',
    'diabetes_prevalence', 'female_smokers', 'male_smokers', 'handwashing_facilities'
]

df = df[columns_to_keep]

In [7]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 404525 entries, 0 to 404524
Data columns (total 40 columns):
 #   Column                               Non-Null Count   Dtype  
---  ------                               --------------   -----  
 0   iso_code                             404525 non-null  object 
 1   continent                            385141 non-null  object 
 2   location                             404525 non-null  object 
 3   date                                 404525 non-null  object 
 4   total_cases                          365721 non-null  float64
 5   new_cases_smoothed                   391990 non-null  float64
 6   total_deaths                         343091 non-null  float64
 7   new_deaths_smoothed                  392368 non-null  float64
 8   people_vaccinated                    277824 non-null  float64
 9   new_people_vaccinated_smoothed       191183 non-null  float64
 10  total_cases_per_million              365420 non-null  float64
 11  new_cases_per

In [8]:
df["income_group"] = ""

for i in range(len(df)):
  gdp_per_capita = df["gdp_per_capita"].iloc[i]
  if gdp_per_capita <= 1145:
    df.loc[i, "income_group"] = "Low income"
  elif gdp_per_capita <= 4515 and gdp_per_capita >= 1146:
    df.loc[i, "income_group"] = "Lower middle income"
  elif gdp_per_capita <= 14005 and gdp_per_capita >= 4516:
    df.loc[i, "income_group"] = "Upper middle income"
  else:
    df.loc[i, "income_group"] = "High income"

df.to_csv("owid-covid-data-preprocesado.csv.gz")

In [9]:
df.to_csv("owid-covid-data-preprocesado.csv.gz")